In [1]:
# Core RAG Frameworks
!pip install -qqqq -U qdrant-client
!pip install -qqqq -U fastembed
!pip install -qqqq -U transformers
!pip install -qqqq -U langchain
!pip install -qqqq -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 1.2 MB/s eta 0:00:00


In [2]:
import json

import numpy as np
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    SparseVector,
    PointStruct,
    QueryRequest,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
    ScoredPoint,
)
from transformers import AutoTokenizer
import json
import os
import time
import fastembed
from fastembed import SparseEmbedding, SparseTextEmbedding, TextEmbedding
from langchain_core.documents import Document  # Document abstraction
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # Prompt templates
from langchain_core.messages import HumanMessage, AIMessage  # Chat history message types
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output
from langchain_core.runnables import RunnablePassthrough  # Pass data through pipeline> lambda x:x
from langchain_openai import  ChatOpenAI
print(f"FastEmbed version: {fastembed.__version__}")

from google.colab import userdata


FastEmbed version: 0.8.0


In [3]:
# ============================================================================
# PART 1: Data Ingestion & Pandas Setup
# ============================================================================
print("\n" + "="*80)
print("Data Ingestion & Pandas Setup")
print("="*80)

# Load tickets
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

# Convert to Pandas DataFrame
df = pd.DataFrame(tickets)

print(f"✓ Created {len(tickets)} documents in pandas")
df.head()


Data Ingestion & Pandas Setup
✓ Loaded 22 support tickets
✓ Created 22 documents in pandas


,ticket_id,title,description,resolution,category,priority,created_date,resolved_date
0,TICK-001,Users unable to log in after password reset,Multiple users reporting authentication failur...,Found that the password hash algorithm was upd...,Authentication,High,2024-01-15,2024-01-15
1,TICK-002,Database connection timeout in production,Application experiencing intermittent 500 erro...,Database connection pool was sized too small f...,Database,Critical,2024-01-18,2024-01-18
2,TICK-003,Payment processing fails for international cards,Customers using non-US credit cards receiving ...,Payment gateway configuration was missing inte...,Payment,High,2024-01-20,2024-01-21
3,TICK-004,Mobile app crashes on iOS 17,App crashes immediately on launch for users wh...,iOS 17 deprecated certain networking APIs we w...,Mobile,Critical,2024-01-22,2024-01-23
4,TICK-005,Memory leak in background worker process,Background job processor showing steady memory...,Memory profiling revealed unclosed database cu...,Performance,High,2024-01-25,2024-01-26


In [4]:

# ============================================================================
# PART 2: Setting up derived columns in Pandas
# ============================================================================
print("\n" + "="*80)
print("Setting up derived columns in Pandas")
print("="*80)

df["combined_text"] = (
    df["title"] + "\n" + df["description"] + "\n" + df["resolution"]
)

# Preview
print(df["combined_text"].iloc[0][:800] + "...")
print(f"\nTotal documents to embed: {len(df)}")


Setting up derived columns in Pandas
Users unable to log in after password reset
Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change....

Total documents to embed: 22


In [5]:
# ============================================================================
# PART 3: Choosing the models for Embedding
# ============================================================================
print("\n" + "="*80)
print("Choosing the models for Embedding")
print("="*80)

# Models we will use
sparse_model_name = "prithvida/Splade_PP_en_v1"
dense_model_name = "BAAI/bge-small-en-v1.5"

# This triggers model download on first run (~500MB sparse + ~1.2GB dense)
sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)
dense_model = TextEmbedding(model_name=dense_model_name, batch_size=32)

print("✅ Both models loaded!")


Choosing the models for Embedding


/tmp/ipykernel_1392/1570556760.py:13: DeprecationWarning: The right spelling is prithivida/Splade_PP_en_v1. Support of this name will be removed soon, please fix the model_name
  sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=32)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Both models loaded!


In [7]:
# ============================================================================
# PART 4: Creating Helper Function that embed the text according to the model
# ============================================================================
print("\n" + "="*80)
print("Creating Helper Function that embed the text according to the model")
print("="*80)

# Helper functions
def make_sparse_embedding(texts: list[str]) -> list[SparseEmbedding]:
    return list(sparse_model.embed(texts, batch_size=32))

def make_dense_embedding(texts: list[str]):
    return list(dense_model.embed(texts))

print("✅ Both Embedding function created!")


Creating Helper Function that embed the text according to the model
✅ Both Embedding function created!


In [10]:
# ============================================================================
# PART 5: Creating columns for holding Dense and Sparse Embedding in Panda
# ============================================================================
print("\n" + "="*80)
print("Creating columns for holding Dense and Sparse Embedding in Panda")
print("="*80)

product_texts = df["combined_text"].tolist()
print(f"Embedding {len(product_texts)} products...")

print("⏳ Generating sparse embeddings (SPLADE)...")
df["sparse_embedding"] = make_sparse_embedding(product_texts)
print("✅ Sparse Embedding Column created !")

print("⏳ Generating dense embeddings (BGE-Large)...")
df["dense_embedding"] = make_dense_embedding(product_texts)
print("✅ Dense Embedding Column created !")






Creating columns for holding Dense and Sparse Embedding in Panda
Embedding 22 products...
⏳ Generating sparse embeddings (SPLADE)...
✅ Sparse Embedding Column created !
⏳ Generating dense embeddings (BGE-Large)...
✅ Dense Embedding Column created !


In [11]:
# ============================================================================
# PART 6: Creating Qdrant Collection and naming it
# ============================================================================
print("\n" + "="*80)
print("Creating Qdrant Collection and naming it ")
print("="*80)

client = QdrantClient(":memory:")  # In-memory for demo; use url="http://localhost:6333" in production
collection_name = "rag_vectorstore"
client.create_collection(
    collection_name,
    # Dense vector config: 1024-dim with cosine similarity
    vectors_config={
        "text-dense": VectorParams(
            size=384,             # Must match model output dimension
            distance=Distance.COSINE,  # Cosine similarity for dense vectors
        )
    },
    # Sparse vector config: no fixed size needed (it's dynamic)
    sparse_vectors_config={
        "text-sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,    # Keep in RAM for speed
            )
        )
    },
)
print(f"✅ Collection '{collection_name}' created with dense + sparse vector support")




Creating Qdrant Collection and naming it 
✅ Collection 'rag_vectorstore' created with dense + sparse vector support


In [12]:
# ============================================================================
# PART 7: Creating a function that returns Points for Qdrant Collection
# ============================================================================
print("\n" + "="*80)
print("Creating a function that returns Points for Qdrant Collection ")
print("""
Each "point" in Qdrant consists of:
                              ID — unique identifier
                              Vectors — our dense + sparse embeddings (stored under named keys)
                              Payload — metadata (product text, product ID, etc.)
""")
print("="*80)

def make_points(df: pd.DataFrame) -> list[PointStruct]:
    """Convert DataFrame rows into Qdrant PointStruct objects."""
    sparse_vectors = df["sparse_embedding"].tolist()
    ticket_texts = df["combined_text"].tolist()
    dense_vectors = df["dense_embedding"].tolist()
    rows = df.to_dict(orient="records")

    points = []
    for idx, (text, sparse_vector, dense_vector) in enumerate(
        zip(ticket_texts, sparse_vectors, dense_vectors)
    ):
        # Convert sparse embedding to Qdrant's SparseVector format
        sparse_vector = SparseVector(
            indices=sparse_vector.indices.tolist(),
            values=sparse_vector.values.tolist()
        )

        point = PointStruct(
            id=idx,
            payload={
                "text": text,
                "ticket_id": rows[idx]["ticket_id"],
            },
            vector={
                "text-sparse": sparse_vector,   # Sparse vector under its named key
                "text-dense": dense_vector.tolist(),  # Dense vector under its named key
            },
        )
        points.append(point)
    return points

print("✅ Qdrant PointStruct objects created !")

# Generate sparse and dense embeddings and add them to the DataFrame
df["sparse_embedding"] = make_sparse_embedding(df["combined_text"].tolist())
df["dense_embedding"] = make_dense_embedding(df["combined_text"].tolist())




Creating a function that returns Points for Qdrant Collection 

Each "point" in Qdrant consists of:
                              ID — unique identifier
                              Vectors — our dense + sparse embeddings (stored under named keys)
                              Payload — metadata (product text, product ID, etc.)

✅ Qdrant PointStruct objects created !


In [13]:
# ============================================================================
# PART 8: Call the function that returns Points and add them to Qdrant Collection
# ============================================================================
print("\n" + "="*80)
print("Call the function that returns Points and add them to Qdrant Collection")
print("\n" + "="*80)

points = make_points(df)
print(f"Prepared {len(points)} points for indexing")

client.upsert(collection_name, points)
print(f"✅ {len(points)} points indexed in Qdrant!")




Call the function that returns Points and add them to Qdrant Collection

Prepared 22 points for indexing
✅ 22 points indexed in Qdrant!


In [17]:
# ============================================================================
# PART 9: Create the search function that does hybrid search
# ============================================================================

print("\n" + "="*80)
print("Create the search function that does hybrid search")
print("\n" + "="*80)

def fetch_hybrid_search(query_text: str, top_k: int = 10):
    """
    Perform hybrid search: run dense AND sparse search in parallel,
    return both result lists.
    """
    # Step 1: Embed the query with BOTH models
    query_sparse_vectors = make_sparse_embedding([query_text])
    query_dense_vector = make_dense_embedding([query_text])

    # Step 2: Fire two searches in a single batch request
    search_results = client.query_batch_points(
        collection_name=collection_name,
        requests=[
            # Search 1: Dense (semantic similarity)
            QueryRequest(
                query=query_dense_vector[0].tolist(),
                using="text-dense",
                limit=top_k,
                with_payload=True,
            ),
            # Search 2: Sparse (lexical matching)
            QueryRequest(
                query=SparseVector(
                    indices=query_sparse_vectors[0].indices.tolist(),
                    values=query_sparse_vectors[0].values.tolist(),
                ),
                using="text-sparse",
                limit=top_k,
                with_payload=True,
            ),
        ],
    )
    # Extract the point lists from QueryResponse objects
    return [search_results[0].points, search_results[1].points]

print(f"✅ fetch_hybrid_search function created thats calls Qdrant")



Create the search function that does hybrid search

✅ fetch_hybrid_search function created thats calls Qdrant


In [18]:

# ============================================================================
# PART 10: Call the search function that does hybrid search
# ============================================================================

print("\n" + "="*80)
print("Call the search function that does hybrid search")
print("\n" + "="*80)


query_text = "Why are emails not being delivered?"
search_results = fetch_hybrid_search(query_text)

print(f"✅ fetch_hybrid_search function called ")

dense_results, sparse_results = search_results[0], search_results[1]

print(f"✅ fetch_hybrid_search function results are fetched ")

print(f"Query: '{query_text}'")
print(f"\n{'='*60}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(dense_results[:5]):
    ticket_id = point.payload['ticket_id']
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] [Ticket Id: {ticket_id}] {title}")

print(f"\n{'='*60}")
print(f"SPARSE (Lexical/SPLADE) Results — Top 5:")
print(f"{'='*60}")
for i, point in enumerate(sparse_results[:5]):
    title = point.payload['text'].split('\n')[0][:80]
    print(f"  {i+1}. [Score: {point.score:.4f}] [Ticket Id: {ticket_id}] {title}")


Call the search function that does hybrid search

✅ fetch_hybrid_search function called 
✅ fetch_hybrid_search function results are fetched 
Query: 'Why are emails not being delivered?'

DENSE (Semantic) Results — Top 5:
  1. [Score: 0.7728] [Ticket Id: TICK-006] Email notifications not being delivered
  2. [Score: 0.6930] [Ticket Id: TICK-022] Email notifications not being already registered
  3. [Score: 0.6207] [Ticket Id: TICK-017] Real-time notifications delayed by several minutes
  4. [Score: 0.5741] [Ticket Id: TICK-018] API returning inconsistent data
  5. [Score: 0.5736] [Ticket Id: TICK-019] Image thumbnails not generating

SPARSE (Lexical/SPLADE) Results — Top 5:
  1. [Score: 19.4827] [Ticket Id: TICK-019] Email notifications not being delivered
  2. [Score: 11.4743] [Ticket Id: TICK-019] Email notifications not being already registered
  3. [Score: 10.5122] [Ticket Id: TICK-019] Webhook deliveries failing intermittently
  4. [Score: 7.1129] [Ticket Id: TICK-019] Users unabl

In [20]:
# ============================================================================
# PART 11: Create the Rank List Function to convert score to ranking And Calling it
# ============================================================================

print("\n" + "="*80)
print("Create the Rank List Function to convert score to ranking And Calling it ")
print("\n" + "="*80)


def rank_list(search_result: list[ScoredPoint]):
    """Convert Qdrant search results to (id, rank) pairs."""
    return [(point.id, point.payload["ticket_id"], rank + 1) for rank, point in enumerate(search_result)]

print(f"✅ Created the Rank List Function to convert score to ranking ")

print(f"✅ Calling the Rank List Function to convert score to ranking ")

print("\n" + "-"*80)

# Convert both result lists to rank lists
dense_rank_list = rank_list(search_results[0])
sparse_rank_list = rank_list(search_results[1])

print("Dense ranks:", dense_rank_list[:5], "...")
print("Sparse ranks:", sparse_rank_list[:5], "...")


Create the Rank List Function to convert score to ranking And Calling it 

✅ Created the Rank List Function to convert score to ranking 
✅ Calling the Rank List Function to convert score to ranking 

--------------------------------------------------------------------------------
Dense ranks: [(5, 'TICK-006', 1), (21, 'TICK-022', 2), (16, 'TICK-017', 3), (17, 'TICK-018', 4), (18, 'TICK-019', 5)] ...
Sparse ranks: [(5, 'TICK-006', 1), (21, 'TICK-022', 2), (12, 'TICK-013', 3), (0, 'TICK-001', 4), (15, 'TICK-016', 5)] ...


In [21]:
# ============================================================================
# PART 12: Create the Reciprocal Rank Fusion function
# ============================================================================

print("\n" + "="*80)
print("Create the Reciprocal Rank Fusion function")
print("\n" + "="*80)


def rrf(rank_lists, alpha=60, default_rank=1000):
    """
    Reciprocal Rank Fusion (RRF).

    Takes multiple rank lists and produces a single fused ranking.
    Only considers RANK POSITION, not raw scores — making it safe to
    combine results from different scoring systems.

    Args:
        rank_lists: List of [(item_id, rank), ...] lists
        alpha: Smoothing constant (default=60, from Cormack et al. 2009)
        default_rank: Rank assigned to items not in a list (high = penalty)

    Returns:
        Sorted list of (item_id, rrf_score) tuples
    """
    all_items = set(item for rank_list in rank_lists for item, _, _ in rank_list)
    item_to_index = {item: idx for idx, item in enumerate(all_items)}

    # Matrix: rows = items, cols = rank lists, filled with default_rank
    rank_matrix = np.full((len(all_items), len(rank_lists)), default_rank)

    for list_idx, rank_list in enumerate(rank_lists):
        for item, ticket_id, rank in rank_list:
            rank_matrix[item_to_index[item], list_idx] = rank

    # RRF formula: sum of 1/(alpha + rank) across all lists
    rrf_scores = np.sum(1.0 / (alpha + rank_matrix), axis=1)

    sorted_indices = np.argsort(-rrf_scores)  # Descending
    sorted_items = [(list(item_to_index.keys())[idx], rrf_scores[idx]) for idx in sorted_indices]

    return sorted_items

print(f"✅ Created the Reciprocal Rank Fusion function")


Create the Reciprocal Rank Fusion function

✅ Created the Reciprocal Rank Fusion function


In [24]:
# ============================================================================
# PART 13: Call the Reciprocal Rank Fusion function
# ============================================================================

print("\n" + "="*80)
print("Call the Reciprocal Rank Fusion function")
print("\n" + "="*80)

print(f"✅ Calling the Reciprocal Rank Fusion function")

# Fuse the rankings!
rrf_rank_list = rrf([dense_rank_list, sparse_rank_list])

print(f"\n🏆 Reciprocal Rank Fusion: '{rrf_rank_list}'")




Call the Reciprocal Rank Fusion function

✅ Calling the Reciprocal Rank Fusion function

🏆 Reciprocal Rank Fusion: '[(5, np.float64(0.03278688524590164)), (21, np.float64(0.03225806451612903)), (12, np.float64(0.030798389007344232)), (16, np.float64(0.03057889822595705)), (15, np.float64(0.030090497737556562)), (2, np.float64(0.028985507246376812)), (17, np.float64(0.016568396226415094)), (0, np.float64(0.016568396226415094)), (18, np.float64(0.01632801161103048)), (14, np.float64(0.016094911377930246)), (11, np.float64(0.016094911377930246)), (7, np.float64(0.015868769360743454)), (10, np.float64(0.01522911051212938)), (19, np.float64(0.01522911051212938))]'


In [25]:
# ============================================================================
# PART 14: Call the Retrieve Method of Qdrant
# ============================================================================

print("\n" + "="*80)
print("Call the Retrieve Method of Qdrant")
print("\n" + "="*80)

# Retrieve full records for the fused results
full_fused_records = client.retrieve(
    collection_name=collection_name,
    ids=[item[0] for item in rrf_rank_list]
)

print(f"✅ Called the Retrieve Method of Qdrant")

top_3_fused_records = client.retrieve(
        collection_name=collection_name,
        ids=[item[0] for item in rrf_rank_list[:3]],
    )

print("full_fused_records")
print("\n" + "-"*80)
for full_fused_record in full_fused_records:
    print(full_fused_record)
print("\n" + "-"*80)
# print("record_by_id")
# Create a lookup by ID
full_record_by_ids = {r.id: r for r in full_fused_records}


for rank, (item_id, score) in enumerate(rrf_rank_list, 1):
    record = full_record_by_ids[item_id]
    ticket_id = record.payload['ticket_id']
    # print(ticket_id)
    title = record.payload['text'].split('\n')[0][:75]
    # print(title)

# Check if this item was in dense, sparse, or both
    in_dense = any(id == item_id for id, _, _ in dense_rank_list)
    in_sparse = any(id == item_id for id, _, _ in sparse_rank_list)
    source = "BOTH" if (in_dense and in_sparse) else ("Dense only" if in_dense else "Sparse only")

    print(f"  {rank:>2}. [{source:<12}] <{ticket_id}> {title}")


Call the Retrieve Method of Qdrant

✅ Called the Retrieve Method of Qdrant
full_fused_records

--------------------------------------------------------------------------------
id=5 payload={'text': "Email notifications not being delivered\nUsers reporting they're not receiving password reset and confirmation emails. SMTP server logs show successful sends, but emails never arrive in inbox or spam folder.\nEmail service provider had blacklisted our sending IP due to spam complaints. Implemented SPF, DKIM, and DMARC records. Switched to dedicated IP address with proper warmup process. Added email deliverability monitoring.", 'ticket_id': 'TICK-006'} vector=None shard_key=None order_value=None
id=21 payload={'text': 'Email notifications not being already registered\nUsers complaining that email used for registeration is failing.\nEmail service provider had aalready registered . To reregister they need delete the account and then register', 'ticket_id': 'TICK-022'} vector=None shard_key=No

In [26]:
# ============================================================================
# PART 15: Initialize LLM
# ============================================================================
print("\n" + "="*80)
print("Initializing Language Model")
print("="*80)

os.environ["OPENAI_API_KEY"]  = userdata.get('OPENAI_APIKEY')

# Check if OpenAI key is available
if os.getenv("OPENAI_API_KEY"):
    print("✓ OpenAI API key found")
    # Initialize ChatOpenAI for generation
    # Reference: https://python.langchain.com/docs/integrations/chat/openai
    llm = ChatOpenAI(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
        # For RAG, use temperature=0 to ensure consistent, factual responses
        # Reference: https://platform.openai.com/docs/guides/text-generation/how-should-i-set-the-temperature-parameter
        timeout=120,  # Increase timeout for slower connections
        max_retries=3,  # Retry on transient failures
    )
    print(f"✓ Using {os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')}")
else:
    print("⚠ OpenAI API key not found!")
    print("  Please set OPENAI_API_KEY environment variable")
    print("  Or use Ollama: ollama pull llama2")
    print("\nFor this demo, we'll show the prompt without generating answers.")
    llm = None


Initializing Language Model
✓ OpenAI API key found
✓ Using gpt-4o-mini


In [27]:
# ============================================================================
# PART 16: Create Prompt Template with Anti-Hallucination Rules
# ============================================================================
print("\n" + "="*80)
print("Prompt Engineering for RAG")
print("="*80)

# Define strict grounding prompt
# Prompt engineering is CRUCIAL for RAG - it tells the LLM how to use the context
# Reference: https://platform.openai.com/docs/guides/prompt-engineering
# Key principles:
# 1. Be explicit about using ONLY the provided context
# 2. Define what to do when information is missing
# 3. Request citations for transparency and verification
# 4. Set the role/persona for appropriate tone
prompt_template = """You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.

CRITICAL RULES:
1. Answer using ONLY information from the provided context.
2. If the question is broad or underspecified, provide the best matching known issue(s) from context and state any assumptions.
3. If context is partially relevant, still provide the most likely troubleshooting guidance from relevant tickets.
4. If the answer is truly not present in context, say "I don't have enough information in the ticket history to answer that question."
5. DO NOT make up information or use external knowledge.
6. Always cite ticket IDs for every issue/resolution you mention.
7. If multiple tickets are relevant, summarize each briefly.

Context from support tickets:
{context}

Question: {question}

Helpful Answer (with ticket citations):"""

# Convert string template to ChatPromptTemplate
# This creates a reusable template with variable placeholders
# Reference: https://python.langchain.com/docs/modules/model_io/prompts/
PROMPT = ChatPromptTemplate.from_template(prompt_template)

print("✓ Prompt template created with anti-hallucination rules:")
print("\n" + "-"*80)
print(prompt_template)
print("-"*80)




Prompt Engineering for RAG
✓ Prompt template created with anti-hallucination rules:

--------------------------------------------------------------------------------
You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.

CRITICAL RULES:
1. Answer using ONLY information from the provided context.
2. If the question is broad or underspecified, provide the best matching known issue(s) from context and state any assumptions.
3. If context is partially relevant, still provide the most likely troubleshooting guidance from relevant tickets.
4. If the answer is truly not present in context, say "I don't have enough information in the ticket history to answer that question."
5. DO NOT make up information or use external knowledge.
6. Always cite ticket IDs for every issue/resolution you mention.
7. If multiple tickets are relevant, summarize each briefly.

Context from support tickets:
{context}

Question: {question

In [28]:
# ============================================================================
# PART 17: Create Format Docs function
# ============================================================================
print("\n" + "="*80)
print("Create Format Docs function")
print("="*80)

# Helper function to format retrieved documents
# This concatenates all retrieved document contents into a single context string
def format_docs(docs):
    """
    Convert a list of Qdrant Record objects into a single context string.

    Why this helper exists:
    - Retrievers (client.retrieve) return `List[Record]` objects.
    - Prompt templates expect plain strings for `{context}`.
    - Joining with separators keeps boundaries visible to the LLM.
    """
    # Keep document boundaries explicit so the model can attribute facts by chunk.
    # Docs are Qdrant Record objects, so extract text from payload.
    return "\n\n---\n\n".join([doc.payload['text'] for doc in docs])

print("✓ Create Format Docs function")


Create Format Docs function
✓ Create Format Docs function


In [29]:
# ============================================================================
# PART 18: Build RAG Chain using LCEL (LangChain Expression Language)
# ============================================================================
print("\n" + "="*80)
print("Build RAG Chain using LCEL ")
print("="*80)

# Helper function to dynamically retrieve and rank documents for any query
def retrieve_context(query_text: str) -> str:
    """
    Dynamically executes hybrid search, ranks results using RRF, retrieves
    the top 3 records from Qdrant, and formats them into a single string.
    """
    # 1. Fetch Hybrid search results (Dense + Sparse)
    search_results = fetch_hybrid_search(query_text)

    # 2. Convert raw scoring to rank lists
    dense_ranks = rank_list(search_results[0])
    sparse_ranks = rank_list(search_results[1])

    # 3. Fuse the rankings using RRF
    fused_ranks = rrf([dense_ranks, sparse_ranks])

    # 4. Retrieve top 3 records from the Qdrant DB
    top_3_records = client.retrieve(
        collection_name=collection_name,
        ids=[item[0] for item in fused_ranks[:3]],
    )

    # 5. Format to plain text
    return format_docs(top_3_records)

if llm:
    # Build RAG chain using LCEL (LangChain Expression Language)
    # Now, the 'context' key uses the dynamic 'retrieve_context' function
    # which triggers vector database queries based on the incoming user question.
    qa_chain = (
        {
            "context": RunnablePassthrough() | retrieve_context,
            "question": RunnablePassthrough()
        }
        | PROMPT
        | llm
        | StrOutputParser()
    )
    print("✓ Dynamic RAG chain assembled:")
    print("  Query → Dynamic Retrieval → Context Injection → LLM → Answer")
    print("\nThis is the complete RAG pipeline! Query in → Answer out")
else:
    qa_chain = None
    print("⚠ LLM not available, showing architecture only")


Build RAG Chain using LCEL 
✓ Dynamic RAG chain assembled:
  Query → Dynamic Retrieval → Context Injection → LLM → Answer

This is the complete RAG pipeline! Query in → Answer out


In [30]:
# ============================================================================
# PART 19: Test the RAG System
# ============================================================================
print("\n" + "="*80)
print("Testing the RAG System")
print("="*80)

test_queries = [
    "How do I fix authentication failures after password reset?",
    "How do I make the perfect pizza?"  # Should refuse to answer!
]

for query in test_queries:
    print("\n" + "="*80)
    print(f"QUERY: {query}")
    print("="*80)

    # Show retrieved context
    docs = top_3_fused_records
    print(f"\nRetrieved {len(docs)} relevant tickets:")
    for i, doc in enumerate(docs, 1):
        print(f"\n  [{i}] {doc.payload['ticket_id']}: {doc.payload['text']}")



    if qa_chain:
        # Generate answer
        print("\nGenerating answer...")
        result = qa_chain.invoke(query)

        print("\n" + "-"*80)
        print("ANSWER:")
        print("-"*80)
        print(result)


    else:
        print("\n(LLM not configured - would generate answer here)")


Testing the RAG System

QUERY: How do I fix authentication failures after password reset?

Retrieved 3 relevant tickets:

  [1] TICK-006: Email notifications not being delivered
Users reporting they're not receiving password reset and confirmation emails. SMTP server logs show successful sends, but emails never arrive in inbox or spam folder.
Email service provider had blacklisted our sending IP due to spam complaints. Implemented SPF, DKIM, and DMARC records. Switched to dedicated IP address with proper warmup process. Added email deliverability monitoring.

  [2] TICK-022: Email notifications not being already registered
Users complaining that email used for registeration is failing.
Email service provider had aalready registered . To reregister they need delete the account and then register

  [3] TICK-013: Webhook deliveries failing intermittently
Outgoing webhooks to customer endpoints timing out. Retry logic exhausting all attempts. Some webhooks succeed while others fail for sa